# 🏺 Egyptian Hieroglyph Recognition Pipeline — v3
### Fuentes-Ferrer et al. (2025) — Applied Soft Computing
#### **Notebook 2/2** — Classification (CVV) + Inference

> **يتطلب:** تشغيل Notebook 1 أولاً عشان `Glyph2025_processed/` و `segmented_glyphs/` يكونوا جاهزين.

---


In [ ]:
# ── Shared Imports & Config (من Notebook 1) ──────────────────
import os, random, json, time, warnings, shutil, subprocess, tarfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torchvision.transforms as T
from PIL import Image
from collections import Counter
from dataclasses import dataclass, field
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed()

WORK_DIR      = '/kaggle/working'
GLYPH2025_DIR = os.path.join(WORK_DIR, 'Glyph2025_processed')
IMG_SIZE      = 256
IMG_EXTS      = ('.jpg','.jpeg','.png','.bmp','.webp')
MEAN          = [0.485, 0.456, 0.406]
STD           = [0.229, 0.224, 0.225]
device        = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅  Device : {device}')


### Cell 3.1 — Classification Imports & Data Split

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, classification_report,
    confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.cm as mpl_cm

# Paper split: 90% train+val, 10% test
trainval_df, test_df = train_test_split(df, test_size=0.10, random_state=SEED, stratify=df["y"])
trainval_df = trainval_df.reset_index(drop=True)
test_df     = test_df.reset_index(drop=True)

# CVV: 5 disjoint validation slots (k=5 from paper)
skf         = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_splits  = list(skf.split(trainval_df["path"], trainval_df["y"]))

print(f"Train+Val : {len(trainval_df):,} | Test : {len(test_df):,}")
print(f"CVV Folds : {len(fold_splits)} | Classes : {len(classes)}")


### Cell 3.2 — Augmentation & TTA Transforms

In [ ]:
train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),    # Paper: hieroglyphs appear in both orientations
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(degrees=20),
    T.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.80,1.20), shear=10),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    T.GaussianBlur(kernel_size=3, sigma=(0.1,2.0)),
    T.RandomGrayscale(p=0.15),
    T.ToTensor(), T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.3, scale=(0.02,0.15)),
])

eval_tf = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN,STD)])

# 7 TTA views (improvement over paper's 5)
tta_tfs = [
    eval_tf,
    T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.RandomHorizontalFlip(p=1.0),
               T.ToTensor(), T.Normalize(MEAN,STD)]),
    T.Compose([T.Resize((int(IMG_SIZE*1.15),int(IMG_SIZE*1.15))), T.CenterCrop(IMG_SIZE),
               T.ToTensor(), T.Normalize(MEAN,STD)]),
    T.Compose([T.Resize((int(IMG_SIZE*0.85),int(IMG_SIZE*0.85))),
               T.Pad(int(IMG_SIZE*0.075),fill=0), T.Resize((IMG_SIZE,IMG_SIZE)),
               T.ToTensor(), T.Normalize(MEAN,STD)]),
    T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.RandomRotation(degrees=(10,10)),
               T.ToTensor(), T.Normalize(MEAN,STD)]),
    T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.RandomRotation(degrees=(-10,-10)),
               T.ToTensor(), T.Normalize(MEAN,STD)]),
    T.Compose([T.Resize((int(IMG_SIZE*1.08),int(IMG_SIZE*1.08))), T.CenterCrop(IMG_SIZE),
               T.ToTensor(), T.Normalize(MEAN,STD)]),
]
print(f"✅  Transforms ready | TTA views: {len(tta_tfs)}")


### Cell 3.3 — Dataset, Balancing & Model

In [ ]:
class ImgDS(Dataset):
    def __init__(self, df, transform):
        self.df=df.reset_index(drop=True); self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        return self.transform(Image.open(self.df.loc[idx,"path"]).convert("RGB")), int(self.df.loc[idx,"y"])

class ImgDS_TTA(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        return Image.open(self.df.loc[idx,"path"]).convert("RGB"), int(self.df.loc[idx,"y"])

def collate_pil(batch):
    imgs,ys=zip(*batch); return list(imgs),torch.tensor(ys)

def balance_train_df(tr_df, seed=SEED):
    """Cap = 2x mean — paper augmentation strategy."""
    rng=np.random.default_rng(seed); cap=int(np.floor(2*tr_df["y"].value_counts().mean()))
    parts=[]
    for _,g in tr_df.groupby("y"):
        idx=rng.choice(g.index.to_numpy(), size=cap, replace=(len(g)<cap)); parts.append(tr_df.loc[idx])
    return pd.concat(parts).sample(frac=1.0,random_state=seed).reset_index(drop=True)

def build_model(num_classes: int, dropout: float=0.30) -> nn.Module:
    """ConvNeXt-Tiny pretrained on ImageNet (paper backbone)."""
    m=models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    in_f=m.classifier[-1].in_features
    m.classifier[-1]=nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_f,num_classes))
    return m

def mixup_data(x, y, alpha=0.3):
    lam=np.random.beta(alpha,alpha) if alpha>0 else 1.0
    idx=torch.randperm(x.size(0),device=x.device)
    return lam*x+(1-lam)*x[idx], y, y[idx], lam

def mixup_criterion(crit, pred, ya, yb, lam):
    return lam*crit(pred,ya)+(1-lam)*crit(pred,yb)

m_test=build_model(len(classes))
print(f"✅  ConvNeXt-Tiny: {sum(p.numel() for p in m_test.parameters()):,} parameters")
del m_test


### Cell 3.4 — ✅ Fix: Class Weights for Imbalanced Dataset

In [ ]:
def compute_class_weights(df_train: pd.DataFrame, num_classes: int,
                          device: str) -> torch.Tensor:
    """
    ✅ Fix: class_weights added to CrossEntropyLoss.
    Fixes the class imbalance shown in the dataset dashboard.
    Formula: weight_c = total_samples / (num_classes * count_c)
    Rare classes (few images) get higher weight → model pays more attention to them.
    """
    counts = df_train["y"].value_counts().sort_index()
    weights = []
    total = len(df_train)
    for i in range(num_classes):
        c = counts.get(i, 1)
        weights.append(total / (num_classes * c))
    w = torch.tensor(weights, dtype=torch.float32).to(device)
    # Normalize so mean weight = 1 (keeps the LR scale stable)
    w = w / w.mean()
    return w

# Compute once from trainval_df
class_weights = compute_class_weights(trainval_df, len(classes), device)
print(f"✅  Class weights computed")
print(f"   Min weight : {class_weights.min().item():.3f}  (most common class)")
print(f"   Max weight : {class_weights.max().item():.3f}  (rarest class)")
print(f"   Mean weight: {class_weights.mean().item():.3f}  (normalized)")

# Sanity check: plot weight distribution
fig,ax = plt.subplots(figsize=(18,3))
ax.bar(range(len(class_weights)), class_weights.cpu().numpy(),
       color=plt.cm.RdYlGn(1 - class_weights.cpu().numpy()/class_weights.max().item()))
ax.set_title("Class Weights Distribution (red=rare classes get more weight)")
ax.set_xlabel("Class index"); ax.set_ylabel("Weight")
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR,"class_weights.png"),dpi=100); plt.show()


### Cell 3.5 — Evaluation Helpers

In [ ]:
@torch.no_grad()
def eval_f1_macro(model, loader):
    model.eval(); ys,preds=[],[]
    for x,y in loader:
        preds.extend(model(x.to(device)).argmax(1).cpu().tolist()); ys.extend(y.tolist())
    _,_,f1,_=precision_recall_fscore_support(ys,preds,average="macro",zero_division=0)
    return float(f1)

@torch.no_grad()
def eval_acc(model, loader):
    model.eval(); ys,preds=[],[]
    for x,y in loader:
        preds.extend(model(x.to(device)).argmax(1).cpu().tolist()); ys.extend(y.tolist())
    return accuracy_score(ys,preds)

def print_metrics(y_true, y_pred, title):
    acc=accuracy_score(y_true,y_pred); bacc=balanced_accuracy_score(y_true,y_pred)
    prec,rec,f1,_=precision_recall_fscore_support(y_true,y_pred,average="macro",zero_division=0)
    bar="═"*52
    print(f"\n{bar}\n  {title}\n{bar}")
    print(f"  Accuracy          : {acc:.4f}")
    print(f"  Balanced Accuracy : {bacc:.4f}")
    print(f"  Precision (macro) : {prec:.4f}")
    print(f"  Recall    (macro) : {rec:.4f}")
    print(f"  F1        (macro) : {f1:.4f}")
    return dict(acc=acc,bacc=bacc,prec=prec,rec=rec,f1=f1)

print("✅  Helpers ready")


### Cell 3.6 — Train One CVV Fold (✅ class_weights in loss)

In [ ]:
def train_one_fold(fold_id: int, tr_idx, va_idx,
                   epochs: int=25, patience: int=6, bs: int=64, lr: float=2e-4):
    """
    CVV Training (paper):
    - Backbone frozen 2 epochs → gradual unfreezing
    - MixUp 50% probability
    - CosineAnnealingLR + AMP (mixed precision)
    - Early stopping on val_F1 (not val_loss → avoids overfitting)
    - ✅ Fix: class_weights in CrossEntropyLoss for rare classes
    """
    set_seed(SEED+fold_id)
    tr_df  = trainval_df.iloc[tr_idx].reset_index(drop=True)
    va_df  = trainval_df.iloc[va_idx].reset_index(drop=True)
    tr_bal = balance_train_df(tr_df, seed=SEED+fold_id)

    tr_l=DataLoader(ImgDS(tr_bal,train_tf), batch_size=bs,shuffle=True, num_workers=2,pin_memory=True)
    va_l=DataLoader(ImgDS(va_df, eval_tf),  batch_size=bs,shuffle=False,num_workers=2,pin_memory=True)

    model = build_model(len(classes)).to(device)

    # ✅ Fix: class_weights passed to loss — tackles imbalance
    crit = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.10)

    hp = list(model.classifier.parameters())
    bp = [p for n,p in model.named_parameters() if "classifier" not in n]
    opt = optim.AdamW([{"params":hp,"lr":lr,"weight_decay":0.10},
                        {"params":bp,"lr":lr*0.1,"weight_decay":0.05}])
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    scaler = GradScaler("cuda")
    best_f1=-1.0; best_acc=-1.0; bad=0
    save_path = os.path.join(WORK_DIR, f"model_fold{fold_id}.pt")
    ll,ta_l,va_l2,vf_l = [],[],[],[]

    def sbg(req):
        for n,p in model.named_parameters():
            if "classifier" not in n: p.requires_grad_(req)
    sbg(False)   # freeze backbone initially

    for ep in range(1, epochs+1):
        if ep==3: sbg(True); print(f"[Fold {fold_id}] ── Backbone unfrozen @ ep {ep} ──")
        model.train(); ep_l=[]; cp,ct=[],[]

        for x,y in tqdm(tr_l, desc=f"Fold {fold_id} Ep {ep:02d}", leave=False):
            x,y = x.to(device), y.to(device)
            um = random.random() < 0.5
            opt.zero_grad(set_to_none=True)
            if um:
                xm,ya,yb,lam = mixup_data(x,y,alpha=0.3)
                with autocast(device_type="cuda"): loss=mixup_criterion(crit,model(xm),ya,yb,lam)
            else:
                with autocast(device_type="cuda"): logits=model(x); loss=crit(logits,y)
                cp.extend(logits.argmax(1).cpu().tolist()); ct.extend(y.cpu().tolist())
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(opt); scaler.update()
            ep_l.append(loss.item())

        sched.step()
        avg_l = float(np.mean(ep_l))
        ta    = accuracy_score(ct,cp) if ct else 0.0
        va    = eval_acc(model, va_l)
        vf    = eval_f1_macro(model, va_l)
        print(f"[Fold {fold_id}] ep={ep:02d}  loss={avg_l:.4f}  "
              f"train_acc={ta:.4f}  val_acc={va:.4f}  val_f1={vf:.4f}  "
              f"lr={sched.get_last_lr()[0]:.2e}")
        ll.append(avg_l); ta_l.append(ta); va_l2.append(va); vf_l.append(vf)

        if vf > best_f1+1e-4:
            best_f1,best_acc,bad = vf,va,0
            torch.save(dict(model=model.state_dict(), classes=classes,
                            fold=fold_id, best_val_f1=best_f1,
                            best_val_acc=best_acc), save_path)
        else:
            bad+=1
            if bad>=patience:
                print(f"[Fold {fold_id}] Early stop. best_val_f1={best_f1:.4f}"); break

    # ── Overfitting check plot ────────────────────────────────
    eps = list(range(1, len(ll)+1))
    fig,axes = plt.subplots(1,3,figsize=(18,4))
    axes[0].plot(eps,ll,marker="o",color="steelblue")
    axes[0].set_title(f"Fold {fold_id} — Loss"); axes[0].set_xlabel("Epoch")

    axes[1].plot(eps,ta_l,marker="o",label="Train",color="steelblue")
    axes[1].plot(eps,va_l2,marker="s",label="Val",color="tomato")
    gap = [t-v for t,v in zip(ta_l,va_l2)]
    axes[1].fill_between(eps, ta_l, va_l2,
                          alpha=0.15, color="red" if max(gap)>0.15 else "green",
                          label=f"Gap max={max(gap):.3f}")
    axes[1].set_title(f"Fold {fold_id} — Accuracy (gap=overfit signal)")
    axes[1].legend(fontsize=8); axes[1].set_ylim(0,1)

    axes[2].plot(eps,vf_l,marker="^",color="mediumseagreen")
    axes[2].axhline(max(vf_l),color="gray",ls="--",alpha=0.5,label=f"Best={max(vf_l):.4f}")
    axes[2].set_title(f"Fold {fold_id} — Val F1 (macro)")
    axes[2].legend(fontsize=8); axes[2].set_ylim(0,1)

    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR,f"fold{fold_id}_curves.png"),dpi=120); plt.show()
    return save_path, best_f1, best_acc, [dict(ep=e,loss=l,val_acc=v,val_f1=f)
                                           for e,l,v,f in zip(eps,ll,va_l2,vf_l)]

print("✅  train_one_fold ready")


### Cell 3.7 — 🚀 CVV Training (5 Folds)

In [ ]:
model_paths=[]; fold_best_f1=[]; fold_best_acc=[]; all_history={}

for fold_id,(tr_idx,va_idx) in enumerate(fold_splits):
    print(f"\n{'═'*55}\n  FOLD {fold_id} / {len(fold_splits)-1}\n{'═'*55}")
    p,bf1,bacc,hist = train_one_fold(fold_id,tr_idx,va_idx)
    model_paths.append(p); fold_best_f1.append(bf1)
    fold_best_acc.append(bacc); all_history[fold_id]=hist

state=dict(model_paths=model_paths, fold_best_f1=fold_best_f1,
           fold_best_acc=fold_best_acc, all_history=all_history,
           classes=classes, class_to_idx=class_to_idx)
with open(os.path.join(WORK_DIR,"training_state.json"),"w") as f: json.dump(state,f,indent=2)

print(f"\n✅  CVV Training complete!")
print(f"   Fold F1s  : {[f'{f:.4f}' for f in fold_best_f1]}")
print(f"   Mean F1   : {np.mean(fold_best_f1):.4f} ± {np.std(fold_best_f1):.4f}")
print(f"   Mean Acc  : {np.mean(fold_best_acc):.4f}")


### Cell 3.8 — CVV Inference (All 4 Voting Methods)

In [ ]:
def load_models(paths):
    ms=[]
    for p in paths:
        ck=torch.load(p,map_location=device,weights_only=True)
        m=build_model(len(classes)).to(device)
        m.load_state_dict(ck["model"]); m.eval(); ms.append(m)
    return ms

ensemble_weights = [f/sum(fold_best_f1) for f in fold_best_f1]
test_loader     = DataLoader(ImgDS(test_df,eval_tf),batch_size=64,shuffle=False,num_workers=2,pin_memory=True)
test_loader_tta = DataLoader(ImgDS_TTA(test_df),batch_size=32,shuffle=False,num_workers=2,pin_memory=True,collate_fn=collate_pil)

@torch.no_grad()
def soft_voting_predict(paths,loader):
    ms=load_models(paths); yt,yp=[],[]
    for x,y in loader:
        x=x.to(device); p=sum(torch.softmax(m(x),dim=1) for m in ms)
        yp.extend((p/len(ms)).argmax(1).cpu().tolist()); yt.extend(y.tolist())
    return np.array(yt),np.array(yp)

@torch.no_grad()
def weighted_soft_voting_predict(paths,loader,weights):
    ms=load_models(paths); yt,yp=[],[]
    for x,y in loader:
        x=x.to(device); ws=sum(w*torch.softmax(m(x),dim=1) for m,w in zip(ms,weights))
        yp.extend(ws.argmax(1).cpu().tolist()); yt.extend(y.tolist())
    return np.array(yt),np.array(yp)

@torch.no_grad()
def weighted_tta_predict(paths,loader_tta,weights):
    ms=load_models(paths); yt,yp=[],[]
    for imgs,y in loader_tta:
        bp=None
        for tf in tta_tfs:
            x=torch.stack([tf(img) for img in imgs]).to(device)
            wp=sum(w*torch.softmax(m(x),dim=1) for m,w in zip(ms,weights))
            bp=wp if bp is None else bp+wp
        yp.extend((bp/len(tta_tfs)).argmax(1).cpu().tolist()); yt.extend(y.tolist())
    return np.array(yt),np.array(yp)

@torch.no_grad()
def hard_voting_predict(paths,loader):
    ms=load_models(paths); yt,yp=[],[]
    for x,y in loader:
        x=x.to(device)
        preds=np.stack([m(x).argmax(1).cpu().numpy() for m in ms])
        final=[np.bincount(preds[:,j]).argmax() for j in range(preds.shape[1])]
        yp.extend(final); yt.extend(y.tolist())
    return np.array(yt),np.array(yp)

print("🔍 Running CVV inference ...")
y_true_sv, y_pred_sv  = soft_voting_predict(model_paths,test_loader)
y_true_wsv,y_pred_wsv = weighted_soft_voting_predict(model_paths,test_loader,ensemble_weights)
y_true_wta,y_pred_wta = weighted_tta_predict(model_paths,test_loader_tta,ensemble_weights)
y_true_hv, y_pred_hv  = hard_voting_predict(model_paths,test_loader)

res_sv  = print_metrics(y_true_sv, y_pred_sv,  "CVV-SV  | Soft Voting")
res_wsv = print_metrics(y_true_wsv,y_pred_wsv, "CVV-WSV | Weighted Soft Voting")
res_wta = print_metrics(y_true_wta,y_pred_wta, "CVV-TTA | Weighted TTA (7 views)")
res_hv  = print_metrics(y_true_hv, y_pred_hv,  "CVV-HV  | Hard Voting")


### Cell 3.9 — 📊 Full Results Dashboard (New Visualization)

In [ ]:
# ── Compute per-class metrics ─────────────────────────────────
_,_,f1_per_class,support = precision_recall_fscore_support(
    y_true_wta, y_pred_wta, average=None, zero_division=0)
per_acc = np.array([
    accuracy_score(y_true_wta[y_true_wta==i], y_pred_wta[y_true_wta==i])
    if (y_true_wta==i).sum()>0 else 0.0
    for i in range(len(classes))
])

fig = plt.figure(figsize=(22, 20))
fig.suptitle("📊 CVV Classification — Full Results Dashboard", fontsize=16, fontweight="bold")

# ── Plot 1: Method Comparison ────────────────────────────────
ax1 = fig.add_subplot(4,2,1)
methods = ["CVV-SV","CVV-WSV","CVV-TTA","CVV-HV"]
accs    = [res_sv["acc"],res_wsv["acc"],res_wta["acc"],res_hv["acc"]]
f1s     = [res_sv["f1"], res_wsv["f1"], res_wta["f1"], res_hv["f1"]]
x = np.arange(len(methods))
ax1.bar(x-0.2, accs, 0.35, label="Accuracy",   color="steelblue")
ax1.bar(x+0.2, f1s,  0.35, label="F1 (macro)", color="tomato")
for i,(a,f) in enumerate(zip(accs,f1s)):
    ax1.text(i-0.2,a+0.005,f"{a:.3f}",ha="center",fontsize=8)
    ax1.text(i+0.2,f+0.005,f"{f:.3f}",ha="center",fontsize=8)
ax1.set_xticks(x); ax1.set_xticklabels(methods,fontsize=9)
ax1.set_ylim(0,1); ax1.set_title("Method Comparison"); ax1.legend(fontsize=9)

# ── Plot 2: CVV Fold Summary ─────────────────────────────────
ax2 = fig.add_subplot(4,2,2)
xf = np.arange(len(fold_best_f1))
ax2.bar(xf-0.2, fold_best_acc, 0.35, label="Val Acc", color="steelblue", alpha=0.85)
ax2.bar(xf+0.2, fold_best_f1,  0.35, label="Val F1",  color="tomato",    alpha=0.85)
ax2.axhline(np.mean(fold_best_f1), color="tomato",    ls="--", lw=1.5,
            label=f"Mean F1={np.mean(fold_best_f1):.4f}")
ax2.axhline(np.mean(fold_best_acc),color="steelblue", ls="--", lw=1.5,
            label=f"Mean Acc={np.mean(fold_best_acc):.4f}")
ax2.set_xticks(xf); ax2.set_xticklabels([f"Fold {i}" for i in range(len(fold_best_f1))])
ax2.set_ylim(0,1); ax2.set_title("CVV Cross-Validation Results"); ax2.legend(fontsize=8)

# ── Plot 3: Per-Class F1 (all classes) ───────────────────────
ax3 = fig.add_subplot(4,2,(3,4))
colors_f1 = plt.cm.RdYlGn(f1_per_class)
bars3 = ax3.bar(classes, f1_per_class, color=colors_f1, edgecolor="none")
ax3.axhline(f1_per_class.mean(), color="dodgerblue", ls="--", lw=1.5,
            label=f"Mean F1 = {f1_per_class.mean():.3f}")
ax3.axhline(0.80, color="orange", ls=":", lw=1, label="F1=0.80 target")
ax3.set_title(f"Per-Class F1 — All {len(classes)} Classes (red=struggling)")
ax3.set_xlabel("Gardiner Code"); ax3.set_ylabel("F1 Score"); ax3.set_ylim(0,1.05)
ax3.legend(fontsize=9); plt.sca(ax3); plt.xticks(rotation=90, fontsize=6)
# Annotate worst 5
worst5 = np.argsort(f1_per_class)[:5]
for wi in worst5:
    ax3.annotate(f"{f1_per_class[wi]:.2f}", (classes[wi], f1_per_class[wi]+0.02),
                 ha="center", fontsize=7, color="red")

# ── Plot 4: Per-Class Accuracy heatmap-style ─────────────────
ax4 = fig.add_subplot(4,2,(5,6))
colors_acc = plt.cm.RdYlGn(per_acc)
ax4.bar(classes, per_acc, color=colors_acc, edgecolor="none")
ax4.axhline(per_acc.mean(), color="dodgerblue", ls="--", lw=1.5,
            label=f"Mean Acc = {per_acc.mean():.3f}")
ax4.set_title(f"Per-Class Accuracy — All {len(classes)} Classes")
ax4.set_xlabel("Gardiner Code"); ax4.set_ylabel("Accuracy"); ax4.set_ylim(0,1.05)
ax4.legend(fontsize=9); plt.sca(ax4); plt.xticks(rotation=90, fontsize=6)

# ── Plot 5: F1 vs Support scatter ────────────────────────────
ax5 = fig.add_subplot(4,2,7)
sc = ax5.scatter(support, f1_per_class, c=f1_per_class,
                  cmap="RdYlGn", alpha=0.7, s=40, vmin=0, vmax=1)
plt.colorbar(sc, ax=ax5, label="F1")
ax5.axhline(0.90, color="orange", ls="--", lw=1, label="F1=0.90")
# Annotate bottom-left (small+bad)
for i,(s,f) in enumerate(zip(support,f1_per_class)):
    if f < 0.60 or s < 15:
        ax5.annotate(classes[i], (s,f), fontsize=6, alpha=0.8)
ax5.set_xlabel("# Test Samples (support)"); ax5.set_ylabel("F1 Score")
ax5.set_title("F1 vs Support (bottom-left = needs more data)")
ax5.legend(fontsize=8)

# ── Plot 6: Top confusions ────────────────────────────────────
ax6 = fig.add_subplot(4,2,8)
wrong = [(y_true_wta[i],y_pred_wta[i]) for i in range(len(y_true_wta)) if y_true_wta[i]!=y_pred_wta[i]]
top10 = Counter(wrong).most_common(10)
if top10:
    conf_labels = [f"{classes[t]}→{classes[p]}" for (t,p),_ in top10]
    conf_vals   = [c for _,c in top10]
    ax6.barh(conf_labels[::-1], conf_vals[::-1], color="tomato", alpha=0.8)
    ax6.set_title("Top 10 Confusion Pairs (CVV-TTA)")
    ax6.set_xlabel("# Wrong Predictions")

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,"results_dashboard.png"),dpi=120,bbox_inches="tight")
plt.show()
print("✅  Results dashboard saved!")


### Cell 3.10 — Confusion Matrix & Grad-CAM

In [ ]:
def plot_cm(y_true, y_pred, title, normalize=None, max_classes=40):
    un,co=np.unique(y_true,return_counts=True); keep=un[np.argsort(-co)][:max_classes]
    mask=np.isin(y_true,keep)
    cm=confusion_matrix(y_true[mask],y_pred[mask],labels=keep,normalize=normalize)
    disp=ConfusionMatrixDisplay(cm,display_labels=[classes[i] for i in keep])
    fig,ax=plt.subplots(figsize=(16,16))
    disp.plot(include_values=False,xticks_rotation=90,ax=ax,colorbar=True)
    ax.set_title(title,fontsize=11); plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR,f"cm_{title[:20].replace(' ','_')}.png"),dpi=100)
    plt.show()

class GradCAM:
    def __init__(self, model):
        self.model=model; self.grad=None; self.act=None
        model.features[-1].register_forward_hook(lambda _,__,o: setattr(self,"act",o.detach()))
        model.features[-1].register_full_backward_hook(lambda _,__,go: setattr(self,"grad",go[0].detach()))
    def __call__(self, tensor, cls=None):
        self.model.eval(); t=tensor.unsqueeze(0).to(device).requires_grad_(True)
        logits=self.model(t)
        if cls is None: cls=logits.argmax(1).item()
        self.model.zero_grad(); logits[0,cls].backward()
        cam=torch.relu((self.grad.mean(dim=[2,3],keepdim=True)*self.act).sum(1)).squeeze().cpu().numpy()
        return (cam-cam.min())/(cam.max()-cam.min()+1e-8), cls

def show_gradcam_grid(model, df_, n=8, seed=SEED):
    rng=np.random.default_rng(seed); idxs=rng.choice(len(df_),size=n,replace=False)
    gcam=GradCAM(model); fig,axes=plt.subplots(2,n,figsize=(n*2.5,6))
    correct=wrong_count=0
    for col,idx in enumerate(idxs):
        row=df_.iloc[idx]; pil=Image.open(row["path"]).convert("RGB")
        cam,pi=gcam(eval_tf(pil)); tl=idx_to_class[int(row["y"])]; pl=idx_to_class[pi]
        correct   += (pl==tl)
        wrong_count+= (pl!=tl)
        axes[0,col].imshow(pil.resize((IMG_SIZE,IMG_SIZE)))
        axes[0,col].set_title(f"True: {tl}",fontsize=7); axes[0,col].axis("off")
        hr=np.array(Image.fromarray(np.uint8(255*cam)).resize((IMG_SIZE,IMG_SIZE)))
        ov=np.clip(0.55*np.array(pil.resize((IMG_SIZE,IMG_SIZE)))/255
                   +0.45*mpl_cm.jet(hr/255)[:,:,:3],0,1)
        axes[1,col].imshow(ov)
        axes[1,col].set_title(f"Pred: {pl}",fontsize=7,
                               color="green" if pl==tl else "red")
        axes[1,col].axis("off")
    plt.suptitle(f"Grad-CAM — {correct}/{n} correct", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR,"gradcam_grid.png"),dpi=120); plt.show()

plot_cm(y_true_wta,y_pred_wta,"Weighted TTA (raw)")
plot_cm(y_true_wta,y_pred_wta,"Weighted TTA (normalized)",normalize="true")

best_idx=int(np.argmax(fold_best_f1))
bm=build_model(len(classes)).to(device)
bm.load_state_dict(torch.load(model_paths[best_idx],map_location=device,weights_only=True)["model"])
show_gradcam_grid(bm,test_df)
print("✅  Visualizations complete!")


### Cell 3.11 — End-to-End: New Stela → Gardiner Codes → NLP Ready

In [ ]:
def predict_stela(image_path: str, show: bool=True,
                   conf_threshold: float=0.30) -> list:
    """
    Full pipeline on a new stela image.
    Step 1: IGSM Segmentation
    Step 2: CVV-TTA Classification
    Step 3: Returns Gardiner codes → feed to NLP model

    conf_threshold: أي glyph الموديل مش واثق فيه (confidence أقل من القيمة دي)
                    بيتجاهله تماماً عشان منحطش حاجة وهمية في الترجمة.
                    Default = 0.30 (30%) — قدّر تعدّل القيمة حسب احتياجك.
    """
    # ── Segmentation ─────────────────────────────────────────
    print(f"\n🔍  Segmenting {os.path.basename(image_path)} ...")
    ordered_masks,crops = run_single(image_path, cfg=seg_cfg,
                                      predictor=sam_predictor, show=show)
    if not crops: print("❌  No glyphs found."); return []

    # ── Classification ────────────────────────────────────────
    print(f"\n🧠  Classifying {len(crops)} glyphs ...")
    ms      = load_models(model_paths)
    results = []
    ignored = 0

    for i,crop in enumerate(crops):
        pil = Image.fromarray(crop.astype(np.uint8)).convert("RGB")
        bp  = None
        for tf in tta_tfs:
            x  = tf(pil).unsqueeze(0).to(device)
            wp = sum(w*torch.softmax(m(x),dim=1) for m,w in zip(ms,ensemble_weights))
            bp = wp if bp is None else bp+wp
        probs = (bp/len(tta_tfs)).squeeze(); conf,pi = probs.max(0)
        conf_val = conf.item()
        code     = idx_to_class[pi.item()]

        # ✅ الفلتر الجديد: لو الموديل مش واثق → تجاهل الـ glyph ده
        if conf_val < conf_threshold:
            print(f"  Glyph {i+1:>3} → ⚠️  IGNORED  (conf={conf_val:.3f} < threshold={conf_threshold:.2f})")
            ignored += 1
            continue

        results.append((crop, code, conf_val))
        print(f"  Glyph {i+1:>3} → {code:>6}  (conf={conf_val:.3f})")

    codes = [r[1] for r in results]
    print(f"\n{'═'*55}")
    print(f"  📊 Stats: {len(crops)} segmented | {len(results)} classified | {ignored} ignored")
    print(f"  📜 Transliteration (Gardiner codes):")
    print(f"     {codes}")
    print(f"\n  📝 Feed 'codes' to your NLP model for translation.")
    print(f"{'═'*55}")

    if show and results:
        n=len(results); cols=min(n,10); rows=(n+cols-1)//cols
        fig,axes=plt.subplots(rows,cols,figsize=(cols*2.2,rows*2.5))
        axes=np.array(axes).flatten()
        for i,(crop,c,conf) in enumerate(results):
            color = plt.cm.RdYlGn(conf)[0:3]
            axes[i].imshow(crop)
            axes[i].set_title(f"{i+1}. {c}\n{conf:.2f}",fontsize=8,
                               color=("green" if conf>0.7 else "orange"))
            axes[i].axis("off")
            for spine in axes[i].spines.values():
                spine.set_edgecolor(color); spine.set_linewidth(2)
        for ax in axes[n:]: ax.axis("off")
        plt.suptitle(
            f"End-to-End Pipeline — {len(results)}/{len(crops)} Glyphs Classified  "
            f"({ignored} ignored, threshold={conf_threshold:.2f})",
            fontsize=13, fontweight="bold"
        )
        plt.tight_layout(); plt.show()
    return results

# ── HOW TO USE ────────────────────────────────────────────────
# results = predict_stela("/path/to/stela.jpg", show=True)
# codes   = [r[1] for r in results]  # → feed to NLP
#
# ✅ لو عايز تبقى أكثر صرامة (تتجاهل أكتر):
# results = predict_stela("/path/to/stela.jpg", conf_threshold=0.50)
#
# ✅ لو عايز تبقى أقل صرامة (تحتفظ بأكتر):
# results = predict_stela("/path/to/stela.jpg", conf_threshold=0.15)

print("\n" + "═"*55)
print("  🏺  PIPELINE COMPLETE — v2 (Fixed + Enhanced)")
print("═"*55)
print("  Step 1 ✅  Data Pre-processing  (Glyph2025 safe)")
print("  Step 2 ✅  Segmentation         (IGSM + SAM + NMS)")
print("  Step 3 ✅  Classification       (CVV + class weights)")
print("  ─────────────────────────────────────────────────")
print("  Fixes : @dataclass | safe resize | subprocess")
print("          class_weights | explicit predictor")
print("  New   : Dataset dashboard | Results dashboard")
print("          Per-class F1/Acc bars | F1 vs Support")
print("          Overfitting gap plot | Top confusions")
print("  ✅ NEW : Confidence filter → ignore ambiguous glyphs")
print("═"*55)


---
# ⚡ ENDPOINT — Session جديدة بدون إعادة التدريب
> شغّل الـ Cell ده بس لو عندك الـ models محفوظة من run سابق.
> **لو بتشغل كل الـ Steps من الأول — مش محتاج الـ Cell ده خالص.**


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  ⚡ ENDPOINT — Load Everything Without Re-running           ║
# ║  استخدم الـ Cell ده في session جديدة بعد ما خلصت           ║
# ║  الـ 3 Steps كلهم مرة واحدة على الأقل                      ║
# ╚══════════════════════════════════════════════════════════════╝

# ── ⚙️  المسار اللي فيه الـ models المحفوظة ──────────────────────
MODELS_DIR = "/kaggle/working"
# لو رفعت الـ models كـ Kaggle Dataset:
# MODELS_DIR = "/kaggle/input/your-hieroglyph-models"

CKPT_PATH = os.path.join(MODELS_DIR, "training_state.json")

# ── 1. Restore Classification State ──────────────────────────────
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(
        f"\n❌  Checkpoint not found: {CKPT_PATH}"
        "\n    → لازم تشغل Steps 1 و 2 و 3 الأول من الأول"
        "\n    → أو رفع الـ models كـ Kaggle Dataset وغيّر MODELS_DIR"
    )

with open(CKPT_PATH) as f:
    state = json.load(f)

model_paths      = [os.path.join(MODELS_DIR, os.path.basename(p)) for p in state["model_paths"]]
fold_best_f1     = state["fold_best_f1"]
fold_best_acc    = state["fold_best_acc"]
all_history      = {int(k):v for k,v in state["all_history"].items()}
classes          = state["classes"]
class_to_idx     = state["class_to_idx"]
idx_to_class     = {int(i):c for c,i in class_to_idx.items()}
ensemble_weights = [f/sum(fold_best_f1) for f in fold_best_f1]

missing = [p for p in model_paths if not os.path.exists(p)]
if missing:
    raise FileNotFoundError("❌  Missing model files:\n" + "\n".join(missing))

print("✅  Classification state restored!")
print(f"   Classes  : {len(classes)}")
print(f"   Fold F1s : {[f'{f:.4f}' for f in fold_best_f1]}")
print(f"   Mean F1  : {np.mean(fold_best_f1):.4f}")

# ── 2. Restore Transforms ─────────────────────────────────────────
eval_tf = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN,STD)])
tta_tfs = [
    eval_tf,
    T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.RandomHorizontalFlip(p=1.0),
               T.ToTensor(), T.Normalize(MEAN,STD)]),
    T.Compose([T.Resize((int(IMG_SIZE*1.15),int(IMG_SIZE*1.15))), T.CenterCrop(IMG_SIZE),
               T.ToTensor(), T.Normalize(MEAN,STD)]),
    T.Compose([T.Resize((int(IMG_SIZE*0.85),int(IMG_SIZE*0.85))),
               T.Pad(int(IMG_SIZE*0.075),fill=0), T.Resize((IMG_SIZE,IMG_SIZE)),
               T.ToTensor(), T.Normalize(MEAN,STD)]),
    T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.RandomRotation(degrees=(10,10)),
               T.ToTensor(), T.Normalize(MEAN,STD)]),
    T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.RandomRotation(degrees=(-10,-10)),
               T.ToTensor(), T.Normalize(MEAN,STD)]),
    T.Compose([T.Resize((int(IMG_SIZE*1.08),int(IMG_SIZE*1.08))), T.CenterCrop(IMG_SIZE),
               T.ToTensor(), T.Normalize(MEAN,STD)]),
]

# ── 3. Reload SegConfig + SAM ─────────────────────────────────────
seg_cfg       = SegConfig(sam_checkpoint=os.path.join(MODELS_DIR,"sam_vit_b.pth"))
sam_predictor = load_sam(seg_cfg)   # ينزل تلقائياً لو مش موجود

# ── 4. Restore model helpers ─────────────────────────────────────
def build_model(num_classes: int, dropout: float=0.30):
    m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    in_f = m.classifier[-1].in_features
    m.classifier[-1] = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_f,num_classes))
    return m

def load_models(paths):
    ms = []
    for p in paths:
        ck = torch.load(p, map_location=device, weights_only=True)
        m  = build_model(len(classes)).to(device)
        m.load_state_dict(ck["model"]); m.eval(); ms.append(m)
    return ms

# ── ✅ READY ──────────────────────────────────────────────────────
print("\n" + "═"*55)
print("  ⚡  ENDPOINT READY — كل حاجة جاهزة!")
print("═"*55)
print(f"  Device    : {device}")
print(f"  Classes   : {len(classes)}")
print(f"  SAM       : ✅  loaded")
print(f"  Models    : {len(model_paths)} folds ✅")
print(f"  Transforms: ✅  eval_tf + {len(tta_tfs)} TTA views")
print()
print("  ✅ دلوقتي تقدر تشغل predict_stela() على أي صورة!")
print("═"*55)
